In [1]:
%matplotlib qt
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("../scripts").resolve()))
import analisis_ttl as ttl
from roi_status_selector import (
    apply_registry_to_processing_tables,
    build_master_roi_outputs,
    load_roi_status_registry,
    prepare_roi_processing_selection,
    save_roi_processing_outputs,
    select_and_update_roi_status,
)

base_dir = Path("/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data")
excel_path = base_dir / "TRPM3_imagenes.xlsx"


### Notebook de processing

Este notebook parte desde los archivos `*_preprocessed_long.csv` generados en `01_preprocessing.ipynb`.

Aquí no se recalculan `phase` ni `trend`: esas marcas ya vienen desde el preprocesamiento. Este segundo paso se usa para:

- cargar las muestras activas del Excel;
- filtrar por genotipo, fase o tendencia;
- seleccionar/excluir ROI con el selector interactivo mediante `ROI_status`;
- graficar y guardar tablas para análisis batch.


In [ ]:
imports = ttl.load_exported_experiments(base_dir)

data_resume = imports["data_resume"]
active_experiments = imports["active_experiments"]
load_status = imports["load_status"]
preprocessed_all = imports["preprocessed_all"]

print(f"experimentos estado A: {len(active_experiments)}")
print(f"preprocessed_all: {preprocessed_all.shape}")

if preprocessed_all.empty:
    print("No se encontraron archivos *_preprocessed_long.csv. Ejecuta primero 01_preprocessing.ipynb para cada muestra.")

active_experiments[["folder", "Genotype", "nick name", "estado", "ROI frames"]]
load_status

experimentos estado A: 8
preprocessed_all: (260100, 38)


,folder,exists,processed_files,filtered_files,preprocessed_files,status
0,mut36_image8,True,[],[],[sample_08_m36_cooling_preprocessed_long.csv],ok
1,mut27_image10,True,[],[],[sample_10_m27_cooling_preprocessed_long.csv],ok
2,mut27_image11,True,[],[],[sample_11_m27_cooling_preprocessed_long.csv],ok
3,mut57_image2,True,[],[],[sample_02_m57_cooling_preprocessed_long.csv],ok
4,mut57_image3,True,[],[],[sample_03_m57_cooling_preprocessed_long.csv],ok
5,mut57_image4,True,[],[],[sample_04_m57_cooling_preprocessed_long.csv],ok
6,mut45_image13,True,[],[],[sample_13_m45_cooling_preprocessed_long.csv],ok
7,mut65_image19,True,[],[],[sample_19_m65_cooling_preprocessed_long.csv],ok


In [5]:
genotype_filter = ["m65"]
phase_filter = "cooling"
trend_filter = "stable"
# Ejemplos:
# genotype_filter = None
# genotype_filter = ["m27", "m36"]
# phase_filter = None
# trend_filter = "increase"  # "increase", "decrease", "stable", "insufficient", None

In [6]:
# Script de procesamiento.
roi_status_id_cols = ["source_folder", "source_file", "sample", "genotype", "genotype_meta", "nickname_meta", "ROI"]
roi_id_cols = roi_status_id_cols
batch_dir = base_dir / "batch_analysis"
batch_dir.mkdir(exist_ok=True)

if "roi_status_registry" not in globals():
    roi_status_registry, roi_status_registry_path = load_roi_status_registry(
        batch_dir,
        id_cols=roi_status_id_cols,
    )
elif "roi_status_registry_path" not in globals():
    roi_status_registry_path = batch_dir / "roi_status_registry.csv"

available_genotypes = sorted(preprocessed_all["genotype_meta"].dropna().astype(str).unique()) if not preprocessed_all.empty else []
print("Genotipos disponibles:", available_genotypes)
print("ROIs en registry acumulado:", roi_status_registry.shape[0])
if not roi_status_registry.empty:
    print(roi_status_registry["ROI_status"].value_counts().sort_index())

temp_ranges = {
    "low": (22, 28),
    "mid": (32, 35),
    "high": (36, 42),
}

processing_tables = prepare_roi_processing_selection(
    preprocessed_all,
    registry=roi_status_registry,
    id_cols=roi_status_id_cols,
    filter_func=ttl.filter_by_values,
    temp_summary_to_long_func=ttl.temp_summary_to_long,
    genotype_filter=genotype_filter,
    phase_filter=phase_filter,
    trend_filter=trend_filter,
    temp_ranges=temp_ranges,
)

processing_selected = processing_tables["processing_selected"]
processing_active = processing_tables["processing_active"]
roi_temp_summary = processing_tables["roi_temp_summary"]
roi_temp_summary_active = processing_tables["roi_temp_summary_active"]
range_long = processing_tables["range_long"]

available_rois_after_filters = sorted(processing_selected["ROI"].dropna().astype(str).unique()) if not processing_selected.empty else []

print("Filtro genotype:", "todos" if genotype_filter is None else genotype_filter)
print("Filtro phase:", "todas" if phase_filter is None else phase_filter)
print("Filtro trend:", "todos" if trend_filter is None else trend_filter)
print(f"processing_selected total: {processing_selected.shape}")
print(f"processing_active ROI_status=1: {processing_active.shape}")
print(f"ROIs disponibles después de filtros: {len(available_rois_after_filters)}")
print("ROI_status en filtro actual:")
print(processing_selected["ROI_status"].value_counts().sort_index())
print(available_rois_after_filters)
print("ROIs resumidas total:", roi_temp_summary.shape[0])
print("ROIs activas ROI_status=1:", roi_temp_summary_active.shape[0])
print("Puntos rango-ROI activos:", range_long.shape[0])
if not roi_temp_summary_active.empty:
    print(roi_temp_summary_active["trend"].value_counts())

roi_temp_summary.head()


Genotipos disponibles: ['m27', 'm36', 'm45', 'm57', 'm65']
ROIs en registry acumulado: 1057
ROI_status
0     96
1    961
Name: count, dtype: int64
Filtro genotype: ['m65']
Filtro phase: cooling
Filtro trend: stable
processing_selected total: (500, 38)
processing_active ROI_status=1: (300, 38)
ROIs disponibles después de filtros: 10
ROI_status en filtro actual:
ROI_status
0    200
1    300
Name: count, dtype: int64
['ROI108', 'ROI14', 'ROI26', 'ROI29', 'ROI35', 'ROI55', 'ROI60', 'ROI8', 'ROI80', 'ROI87']
ROIs resumidas total: 10
ROIs activas ROI_status=1: 6
Puntos rango-ROI activos: 18
trend
stable    6
Name: count, dtype: int64


,source_folder,source_file,sample,genotype,genotype_meta,nickname_meta,ROI,low_mean,low_sd,low_n,mid_mean,mid_sd,mid_n,high_mean,high_sd,high_n,delta_high_low,trend,ROI_status
0,mut65_image19,sample_19_m65_cooling_preprocessed_long.csv,sample_19,m65,m65,image19,ROI8,0.005905,0.002200,4,0.012963,0.003798,10,0.009120,0.005334,7,0.003215,stable,0
1,mut65_image19,sample_19_m65_cooling_preprocessed_long.csv,sample_19,m65,m65,image19,ROI14,-0.003321,0.005861,4,-0.001351,0.003176,10,-0.007174,0.005907,7,-0.003853,stable,1
2,mut65_image19,sample_19_m65_cooling_preprocessed_long.csv,sample_19,m65,m65,image19,ROI26,-0.011961,0.004115,4,-0.014862,0.005841,10,-0.001989,0.003297,7,0.009971,stable,1
3,mut65_image19,sample_19_m65_cooling_preprocessed_long.csv,sample_19,m65,m65,image19,ROI29,-0.001673,0.002843,4,-0.006434,0.002371,10,-0.010698,0.006264,7,-0.009024,stable,1
4,mut65_image19,sample_19_m65_cooling_preprocessed_long.csv,sample_19,m65,m65,image19,ROI35,0.010553,0.001889,4,0.030011,0.005730,10,0.009904,0.008896,7,-0.000649,stable,0


In [7]:
launch_roi_selector = True

if launch_roi_selector:
    roi_temp_summary, roi_status_registry = select_and_update_roi_status(
        roi_temp_summary,
        registry=roi_status_registry,
        id_cols=roi_id_cols,
        registry_path=roi_status_registry_path,
        save_path=batch_dir / "roi_temp_summary_with_roi_status.csv",
    )

    processing_tables = apply_registry_to_processing_tables(
        processing_tables,
        registry=roi_status_registry,
        id_cols=roi_id_cols,
        temp_summary_to_long_func=ttl.temp_summary_to_long,
        temp_ranges=temp_ranges,
    )

    processing_selected = processing_tables["processing_selected"]
    processing_active = processing_tables["processing_active"]
    roi_temp_summary = processing_tables["roi_temp_summary"]
    roi_temp_summary_active = processing_tables["roi_temp_summary_active"]
    range_long = processing_tables["range_long"]

print("ROI_status en resumen:")
print(roi_temp_summary["ROI_status"].value_counts().sort_index())


ROI_status registry actualizado y guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/roi_status_registry.csv
ROI_status en resumen:
ROI_status
0    4
1    6
Name: count, dtype: int64


In [ ]:
#Graph 1
x_pos = {"low": 0, "mid": 1, "high": 2}
x_labels = [f"{name}\n{lo}-{hi}°C" for name, (lo, hi) in temp_ranges.items()]
label_roi_points = True
samples = sorted(range_long["sample"].dropna().unique()) if not range_long.empty else []
colors = dict(zip(samples, plt.cm.tab10(range(len(samples)))))

plt.figure(figsize=(8, 5))

for sample_name, sub in range_long.groupby("sample", dropna=False):
    xs = sub["temp_range"].map(x_pos).astype(float)
    jitter = ((sub.groupby("temp_range").cumcount() % 9) - 4) * 0.012
    plot_x = xs + jitter
    plt.scatter(
        plot_x,
        sub["mean_normsignal"],
        s=22,
        alpha=0.45,
        color=colors.get(sample_name, "0.5"),
        label=sample_name,
    )

    if label_roi_points:
        for x, y, roi in zip(plot_x, sub["mean_normsignal"], sub["ROI"]):
            plt.annotate(
                str(roi),
                (x, y),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=7,
                alpha=0.75,
            )

if not range_long.empty:
    group_summary = range_long.groupby("temp_range", observed=True).agg(
        mean=("mean_normsignal", "mean"),
        sem=("mean_normsignal", lambda x: x.std() / (len(x) ** 0.5)),
        n=("mean_normsignal", "size"),
    )
    group_x = [x_pos[idx] for idx in group_summary.index]
    plt.errorbar(
        group_x,
        group_summary["mean"],
        yerr=group_summary["sem"],
        color="black",
        marker="o",
        linewidth=2.5,
        capsize=4,
        label="mean ± SEM",
    )
else:
    group_summary = pd.DataFrame()

plt.axhline(0, linestyle="--", alpha=0.4)
plt.xticks([0, 1, 2], x_labels)
plt.ylabel("Mean NormSignal per ROI")
plt.xlabel("Temperature range")
plt.title("Active ROIs: response by temperature range")
plt.legend(title="Sample", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

group_summary


In [ ]:
#Graph2
plt.figure(figsize=(7, 4))
for sample_name, sub in roi_temp_summary_active.groupby("sample", dropna=False):
    plt.scatter(
        sub["low_mean"],
        sub["high_mean"],
        s=28,
        alpha=0.55,
        color=colors.get(sample_name, "0.5"),
        label=sample_name,
    )

if not roi_temp_summary_active.empty:
    lims = [
        min(roi_temp_summary_active["low_mean"].min(), roi_temp_summary_active["high_mean"].min()),
        max(roi_temp_summary_active["low_mean"].max(), roi_temp_summary_active["high_mean"].max()),
    ]
    plt.plot(lims, lims, color="black", linestyle="--", alpha=0.5)

plt.xlabel("Low mean NormSignal")
plt.ylabel("High mean NormSignal")
plt.title("Active ROIs: high vs low temperature response")
plt.legend(title="Sample", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

roi_temp_summary_active.groupby(["sample", "trend"]).size().unstack(fill_value=0)


In [ ]:
# Guarda tablas del processing en data/Proc_data/batch_analysis
# Archivo principal para análisis:
# - roi_temp_summary_active.csv contiene TODAS las ROI activas acumuladas.
# Archivos auxiliares:
# - *_current_filter_* conservan la vista del filtro actual.
# - *_all_* conservan el universo completo con ROI_status acumulado.
save_batch_outputs = True

if save_batch_outputs:
    roi_status_registry.to_csv(roi_status_registry_path, index=False)
    print(f"Registry acumulado guardado: {roi_status_registry_path}")

    master_tables = build_master_roi_outputs(
        preprocessed_all,
        registry=roi_status_registry,
        id_cols=roi_status_id_cols,
        temp_summary_to_long_func=ttl.temp_summary_to_long,
        temp_ranges=temp_ranges,
    )

    saved_paths = save_roi_processing_outputs(
        current_tables=processing_tables,
        master_tables=master_tables,
        save_func=ttl.save_batch_dataframe,
        base_dir=base_dir,
    )
